# Probpipe example of simple data assimilation 

In [1]:
from probpipe.core.node import Node, wf
from probpipe.core.workflow_node import WorkflowNode
from probpipe.core.module_node import ModuleNode
from probpipe import MvNormal, Normal1D
from probpipe import EmpiricalDistribution, Distribution
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display



## Prior Node

In [ ]:
class PriorNode(Node):
    def __init__(self):
        super().__init__(child_nodes={}, inputs={})

    def __call__(self) -> Normal1D:
        # β ~ N(0, 10^2)
        return Normal1D(mu=0.0, sigma=10.0)


## Likelihood Node

$$ y_i = x_i \beta + \epsilon_i $$

where \( \epsilon_i \sim \mathcal{N}(0, \sigma^2) \)

and for $\beta$, the likelihood is proportional to the following:
$$ exp(-\frac{1}{2\sigma^2} [(X^TX)\beta^2 - 2(X^T y)\beta] ) $$

In [4]:

class LikelihoodNode(Node):
    def __init__(self):
        super().__init__(child_nodes={}, inputs={})

    def __call__(
        self,
        X: np.ndarray,
        y: np.ndarray,
        sigma2: float,
    ) -> Normal1D:
        XtX = float(X.T @ X)
        Xty = float(X.T @ y)

        # Likelihood as Normal in β (up to proportionality)
        mu_lik = Xty / XtX
        sigma_lik = np.sqrt(sigma2 / XtX)

        return Normal1D(mu=mu_lik, sigma=sigma_lik)


## Assimilation Node
The posterior distribution for $\beta$ is given by:
$$ \sigma^2_n = (\sigma_0^-2 + \sigma_L^-2)^{-1} $$
$$ \mu_n = \sigma^2_n (\frac{\mu_0}{\sigma_0^2}+\frac{\mu_L}{\sigma_L^2}) $$

In [5]:
class AssimilationNode(Node):
    def __init__(self):
        super().__init__(child_nodes={}, inputs={})

    def __call__(
        self,
        prior: Normal1D,
        likelihood: Normal1D,
    ) -> Normal1D:
        var0 = prior.sigma ** 2
        varL = likelihood.sigma ** 2

        var_post = 1.0 / (1.0 / var0 + 1.0 / varL)
        mu_post = var_post * (
            prior.mu / var0 + likelihood.mu / varL
        )

        return Normal1D(mu=mu_post, sigma=np.sqrt(var_post))


## The Regression Module

In [6]:


class RegressionModule(ModuleNode):

    @wf
    def fit(
        self,
        prior: PriorNode,
        likelihood: LikelihoodNode,
        assimilate: AssimilationNode,
        X: np.ndarray,
        y: np.ndarray,
        sigma2: float,
    ) -> Normal1D:
        # Explicit calls
        prior_dist = prior()
        lik_dist = likelihood(X=X, y=y, sigma2=sigma2)
        posterior = assimilate(
            prior=prior_dist,
            likelihood=lik_dist,
        )
        return posterior


In [7]:
prior = PriorNode()
likelihood = LikelihoodNode()
assimilate = AssimilationNode()

model = RegressionModule(
    child_nodes={
        "prior": prior,
        "likelihood": likelihood,
        "assimilate": assimilate,
    },
    inputs={},
    prefect_kind=None,
)


# Execute data assimilation

In [8]:
import numpy as np

np.random.seed(0)

n = 50
X = np.random.randn(n, 1)
true_beta = 2.0
y = X[:, 0] * true_beta + np.random.randn(n) * 0.5

posterior = model.fit(
    X=X,
    y=y,
    sigma2=0.25,
)

print("Posterior mean:", posterior.mu)
print("Posterior sd:", posterior.sigma)


Posterior mean: 1.9763337095329931
Posterior sd: 0.06233927558219409


/var/folders/qw/b5_3_w6j1p93w2z431q7l_s00000gq/T/ipykernel_13899/278308967.py:11: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  XtX = float(X.T @ X)
/var/folders/qw/b5_3_w6j1p93w2z431q7l_s00000gq/T/ipykernel_13899/278308967.py:12: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  Xty = float(X.T @ y)


In [17]:
X1 = np.random.randn(n, 1)
true_beta = 2.0
y1 = X1[:, 0] * true_beta + np.random.randn(n) * 0.5


post1 = model.fit(X=X1, y=y1, sigma2=0.25)


X2 = np.random.randn(n, 1)
true_beta = 2.0
y2 = X2[:, 0] * true_beta + np.random.randn(n) * 0.5


# Use posterior as new prior
post2 = assimilate(
    prior=post1,
    likelihood=likelihood(X=X2, y=y2, sigma2=0.25),
)

post2

/var/folders/qw/b5_3_w6j1p93w2z431q7l_s00000gq/T/ipykernel_13899/278308967.py:11: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  XtX = float(X.T @ X)
/var/folders/qw/b5_3_w6j1p93w2z431q7l_s00000gq/T/ipykernel_13899/278308967.py:12: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  Xty = float(X.T @ y)


In [15]:
X2

array([[-1.05462846,  0.82024784],
       [ 0.46313033,  0.27909576],
       [ 0.33890413,  2.02104356],
       [-0.46886419, -2.20144129],
       [ 0.1993002 , -0.05060354],
       [-0.51751904, -0.97882986],
       [-0.43918952,  0.18133843],
       [-0.5028167 ,  2.41245368],
       [-0.96050438, -0.79311736],
       [-2.28862004,  0.25148442],
       [-2.01640663, -0.53945463],
       [-0.27567053, -0.70972797],
       [ 1.73887268,  0.99439439],
       [ 1.31913688, -0.88241882],
       [ 1.12859406,  0.49600095],
       [ 0.77140595,  1.02943883],
       [-0.90876325, -0.42431762],
       [ 0.86259601, -2.65561909],
       [ 1.51332808,  0.55313206],
       [-0.04570396,  0.22050766],
       [-1.02993528, -0.34994336],
       [ 1.10028434,  1.29802197],
       [ 2.69622405, -0.07392467],
       [-0.65855297, -0.51423397],
       [-1.01804188, -0.07785476],
       [ 0.38273243, -0.03424228],
       [ 1.09634685, -0.2342158 ],
       [-0.34745065, -0.58126848],
       [-1.63263453,